# Module 1 · Lesson 02: Your First OpenAI API Call

Welcome! In this notebook you will learn how to **communicate with OpenAI's GPT models** through their API.

## What you will learn
1. How to initialize the OpenAI client
2. The anatomy of a **chat completion** request
3. Using **system prompts** to control behaviour
4. How **temperature** affects creativity
5. Building **multi-turn conversations**

---

### Prerequisites
Make sure you have run `01_setup_verification.py` and that your `OPENAI_API_KEY` is set in `.env`.

OpenAI API Key: https://platform.openai.com/api-keys

OpenAI API Pricing: https://openai.com/api/pricing/

In [1]:
# ── Setup ──
#
import os

from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv(Path.cwd().parent / ".env")

from openai import OpenAI

client = OpenAI()  # reads OPENAI_API_KEY from environment

if client: 
    display(Markdown("Client ready"))

Client ready

---
## 1. Basic Completion — Your Very First Call

The `chat.completions.create()` method is the **core building block** of every LLM application.

Three required parameters:
| Parameter | Purpose |
|-----------|--------|
| `model`   | Which GPT model to use |
| `messages` | The conversation so far (list of dicts) |
| `max_tokens` | Maximum length of the response |

Each message has a `role` (`"system"`, `"user"`, or `"assistant"`) and `content`.

In [2]:
# ── Example 1: Basic completion ────────────────────────
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "What is Python? Answer in one sentence."}
    ],
    max_tokens=100
    
)

# Extract the answer
answer = response.choices[0].message.content
display(Markdown(f"**Response:** {answer}"))

# Token usage
u = response.usage
print(f"\n Tokens — Prompt: {u.prompt_tokens}, Completion: {u.completion_tokens}, Total: {u.total_tokens}")

**Response:** Python is a high-level, interpreted programming language known for its readability and versatility, making it suitable for various applications such as web development, data analysis, artificial intelligence, and automation.


 Tokens — Prompt: 16, Completion: 36, Total: 52


> **Key Insight:** The API is *stateless* — it does not remember previous calls.
> Every request must contain the full conversation context.

---
## 2. System Prompts — Controlling Behaviour

A **system prompt** sets the AI's persona, tone, and constraints *before* the user's message.
Think of it as the "instruction manual" the model follows.

In [3]:
# ── Example 2: System prompt ──
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": (
                "You are a helpful programming tutor. "
                "Explain concepts simply using analogies. "
                "Keep responses concise (max 3 sentences)."
            )
        },
        {"role": "user", "content": "What is recursion?"}
    ],
    max_tokens=200
)

display(Markdown(f"### Tutor Response\n\n{response.choices[0].message.content}"))

### Tutor Response

Recursion is like a set of nesting dolls; each doll contains a smaller version of itself until you reach the tiniest one. In programming, a recursive function calls itself to solve smaller parts of the problem until it reaches a base case, which stops the process. It's a way for a function to break down complex tasks into simpler, more manageable steps.

> **Best Practice:** Always include a system prompt in production.
> It improves consistency, safety, and output quality.

---
## 3. Temperature — Creativity vs Determinism

| Temperature | Behaviour | Use Case |
|-------------|-----------|----------|
| `0.0` | Deterministic, repeatable | Classification, extraction, math |
| `0.3–0.7` | Balanced | General Q&A, summarisation |
| `1.0` | Creative, varied | Brainstorming, storytelling |

Let's compare:

In [10]:
# ── Example 3: Temperature comparison ──
# Run this cell multiple times — temperature 0 gives the same result, temperature 1 varies!
# prompt = "Write a one-sentence story about a robot."
prompt = "Γράψε μου μια σύντομη ιστορία, το πολύ 2 προτάσεις, για ένα ρομπότ που ήθελε να μάθει Python"

for temp in [0.0, 1.0]:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=160,
        temperature=temp
    )
    label = "Deterministic" if temp == 0.0 else "Creative"
    display(Markdown(f"**Temperature {temp} ({label}):** {response.choices[0].message.content}"))

**Temperature 0.0 (Deterministic):** Ένα ρομπότ ονόματι Ρομπί, που ονειρευόταν να δημιουργήσει τις δικές του εφαρμογές, αποφάσισε να μάθει Python, αλλά κάθε φορά που προσπαθούσε να γράψει κώδικα, οι εντολές του μετατρέπονταν σε αστεία αστεία. Τελικά, ανακάλυψε ότι η δημιουργικότητα και η διασκέδαση ήταν το κλειδί για να γίνει ο καλύτερος προγραμματιστής ρομπότ!

**Temperature 1.0 (Creative):** Ενα ρομπότ με όνειρο να δημιουργήσει την τέλεια εφαρμογή αποφάσισε να μάθει Python, αλλά κάθε φορά που προσπαθούσε να γράψει κώδικα, αντί για кωδικοποίηση έβρισκε μόνο bugs. Τελικά, με τη βοήθεια ενός ανθρώπινου φίλου του, κατάφερε να μετατρέψει τα λάθη του σε ευκαιρίες μάθησης και έγινε ο καλύτερος προγραμματιστής της ρομποτικής του παρέας.

---
## The Problem: LLMs Have NO Memory!

Before we learn the multi-turn pattern, let's **prove** that the API has **no memory**.

Each API call is completely independent. The model doesn't know what you asked before.
Watch what happens when we make two separate calls:

In [11]:
# --- Proof: The API has NO memory between calls ---
# The model does NOT remember previous API calls information
# Each API call is STATELESS -- it has zero memory of previous calls. 
# To have a conversation, YOU must send the full chat history every time@

# Call 1: Tell the model our name
prompt1 = "My name is Alice and I am a software engineer."
response1 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": prompt1}
    ],
    max_tokens=50
)
print("--- CALL 1 ---")
print(f"User:      {prompt1}")
print(f"Assistant: {response1.choices[0].message.content}")

# Call 2: Ask about our name -- completely separate call!
prompt2 = "What is my name and what do I do?"
response2 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": prompt2}
    ],
    max_tokens=50
)
print("\n--- CALL 2 (separate, no history) ---")
print(f"User:      {prompt2}")
print(f"Assistant: {response2.choices[0].message.content}")

--- CALL 1 ---
User:      My name is Alice and I am a software engineer.
Assistant: Nice to meet you, Alice! As a software engineer, what areas do you specialize in? Are there any projects you're currently working on or technologies you're particularly interested in?

--- CALL 2 (separate, no history) ---
User:      What is my name and what do I do?
Assistant: I don’t have access to personal data about individuals unless it has been shared with me in the course of our conversation. Therefore, I can’t know your name or what you do. If you’d like to share that information or ask a specific question


In [12]:

prompt3 = "Τί καιρό έχει στην Αθήνα σήμερα?"

response3 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": prompt3}
    ],
    max_tokens=150
)

print("\n--- CALL 2 (separate, no history) ---")

print(f"User:      {prompt3}")

print(f"Assistant: {response3.choices[0].message.content}")


--- CALL 2 (separate, no history) ---
User:      Τί καιρό έχει στην Αθήνα σήμερα?
Assistant: Λυπάμαι, αλλά δεν μπορώ να παρέχω πληροφορίες για τον καιρό σε πραγματικό χρόνο. Μπορείτε να ελέγξετε μια έγκυρη μετεωρολογική ιστοσελίδα ή εφαρμογή για ενημερώσεις σχετικά με τον καιρό στην Αθήνα.


---
## 4. Multi-Turn Conversations

Since the API is **stateless**, we must send the full conversation history every time.
The pattern is:

```
messages = [
    {"role": "system", "content": "..."},   # Instructions
    {"role": "user",   "content": "..."},    # User turn 1
    {"role": "assistant", "content": "..."},  # Model reply 1
    {"role": "user",   "content": "..."},    # User turn 2
    ...                                        # And so on
]
```

In [13]:
# ── Example 4: Multi-turn conversation ──
# It is going to remember what we have said if we include the full conversation in 'messages'

prompt1 = "My name is Alice."
messages = [
    {"role": "system", "content": "You are a helpful assistant. Be concise."},
    {"role": "user", "content": prompt1},
]

# Turn 1
response1 = client.chat.completions.create(
    model="gpt-4o-mini", messages=messages, max_tokens=50
)

reply1 = response1.choices[0].message.content
print(f" User:      {prompt1}")
print(f" Assistant: {reply1}\n")

# Add the assistant's reply to history
messages.append({"role": "assistant", "content": reply1})

# Turn 2 — does it remember?
prompt2 = "What is my name?"
messages.append({"role": "user", "content": {prompt2}})
response2 = client.chat.completions.create(
    model="gpt-4o-mini", messages=messages, max_tokens=50
)
print(f" User:      {prompt2}")
print(f" Assistant: {response2.choices[0].message.content}")
print(f"\n It remembered! Because we included the full conversation in 'messages'.")

 User:      My name is Alice.
 Assistant: Hello, Alice! How can I assist you today?



BadRequestError: Error code: 400 - {'error': {'message': "Invalid type for 'messages[3].content[0]': expected an object, but got a string instead.", 'type': 'invalid_request_error', 'param': 'messages[3].content[0]', 'code': 'invalid_type'}}

---
## Streaming Responses

By default, the API waits until the **entire** response is generated before returning it.
With **streaming**, tokens arrive one by one -- just like ChatGPT's typing effect!

| Mode | Behaviour | Use Case |
|------|-----------|----------|
| Normal | Wait for full response | Background jobs, data extraction |
| Streaming | Tokens arrive live | Chat UIs, real-time applications |

To enable streaming, simply add `stream=True`:

In [ ]:
# -- Streaming: tokens arrive one by one -------------------
import time

print("Streaming response:\n")

propmpt = "Explain what streaming means in 3 sentences."
stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant. Be concise."},
        {"role": "user", "content": prompt}
    ],
    max_tokens=150,
    stream=True  # <-- This enables streaming!
)

full_text = ""
start = time.time()

for chunk in stream:
    # Each chunk contains a small piece of the response
    token = chunk.choices[0].delta.content
    if token:  # Can be None for the final chunk
        print(token, end="", flush=True)
        full_text += token

elapsed = time.time() - start
print(f"\n\nTotal: {len(full_text)} chars in {elapsed:.1f}s")
print(f"First token appeared almost instantly vs waiting {elapsed:.1f}s for the full response!")

---
## 5. Exercise — Try It Yourself!

Modify the cell below to:
1. Change the **system prompt** to a different persona (e.g., "You are a Shakespearean poet")
2. Ask the model a question
3. Experiment with different `temperature` values

In [25]:
# ── YOUR TURN ─────────────────────────────────────────
# TODO: Change the system prompt and user message below

my_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a Shakespearean poet."},
        {"role": "user", "content": "Explain what an API is."}
    ],
    max_tokens=200,
    temperature=0.7
)

display(Markdown(my_response.choices[0].message.content))

In realms of code where data flows like streams,  
An API, dear friend, doth weave its dreams.  
An "Application Programming Interface,"  
A bridge, a means, for systems to embrace.  

Like actors on a stage, they do perform,  
With protocols that guide, and rules to form.  
'Tis but a syntax, simple yet profound,  
Where one software doth call, and answers found.  

Through calls and responses, they doth exchange,  
Data and functions, in a dance so strange.  
From web to app, they link with deft design,  
Enabling services to intertwine.  

So think of APIs as liaisons fair,  
Uniting realms of tech with utmost care.  
They grant us access, like a key to gates,  
Unlocking wonders, as our world creates.

---
## Key Takeaways

| Concept | Summary |
|---------|--------|
| **Client** | `OpenAI()` reads the API key from environment automatically |
| **Messages** | List of `{role, content}` dicts -- system, user, assistant |
| **System Prompt** | Sets behaviour/persona -- always include in production |
| **Temperature** | 0 = deterministic, 1 = creative |
| **No Memory** | Separate calls do NOT share context -- you must send the full history |
| **Multi-Turn** | Append assistant replies to `messages` list to maintain conversation |
| **Streaming** | `stream=True` makes tokens arrive one by one for responsive UIs |
| **max_tokens** | Controls maximum response length (and cost!) |

---
**Next:** `03_first_anthropic_call.ipynb` -- Learn Claude's API and compare with OpenAI